# Lab 1 — Tiny VLM Adversarial Cost Challenge

## Overview

Welcome to the ML Security Lab! In this lab, you'll implement adversarial attacks against a Vision-Language Model (VLM) using the custom dataset and a frozen TinyCLIP scorer.

### Objective
Your task is to build an attack function that can manipulate either:
- **Caption tokens** (text modifications)
- **Image pixels** (visual modifications)
- **Both** (multimodal attack)

The goal is to flip the model's decision (match → no-match or vice versa) while minimizing the attack cost.

### Constraints
- **T_MAX = 10**: Maximum token edits per sample
- **P_MAX = 100**: Maximum pixel edits per sample  
- **Q_MAX = 100**: Maximum queries per sample
- **Evaluation**: Public leaderboard (1,000 val pairs) + Private leaderboard (1,000 test pairs)

### Scoring
Your attack will be evaluated based on:
1. **Success Rate**: Percentage of samples where you successfully flip the decision
2. **Cost Efficiency**: Lower total cost (token edits + pixel edits + queries) is better
3. **Attack Budget**: Must stay within the specified limits

In [ ]:
import os
import zipfile

# Check if 'images' directory or 'val_pairs.json' file is missing
if not os.path.exists('images') or not os.path.exists('val_pairs.json'):
    print("Required data not found. Extracting 'data.zip'...")
    
    # Check if 'data.zip' exists
    if os.path.exists('data.zip'):
        with zipfile.ZipFile('data.zip', 'r') as zip_ref:
            zip_ref.extractall()  # Extract all files in the current directory
        print("Extraction complete!")
    else:
        print("Error: 'data.zip' not found. Please ensure the file is in the current directory.")
else:
    print("All required data is already present.")

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
import open_clip
from datasets import load_dataset
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
import json
import random
import os
from typing import List, Tuple, Dict, Optional
import warnings
import requests
from urllib.parse import urlparse
import hashlib
from collections import defaultdict
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

print("All packages imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Data Loading
print("Loading validation data...")

# Load validation pairs from JSON
with open('val_pairs.json', 'r') as f:
    val_pairs = json.load(f)

print(f"Loaded {len(val_pairs)} validation pairs")

# Helper function to load images
def load_image_from_pair(pair: dict) -> Image.Image:
    """Load image from the pair dictionary using image_path"""
    return Image.open(pair['image_path']).convert('RGB')

# Sample a few pairs to verify data loading
print("\nSample validation pairs:")
for i in range(3):
    pair = val_pairs[i]
    print(f"  Image ID: {pair['image_id']}, Path: {pair['image_path']}")
    print(f"  Caption: {pair['caption'][:50]}..., Match: {pair['is_match']}")
    print()

print(f"Data distribution:")
labels = [pair['is_match'] for pair in val_pairs]
print(f"  Match (True): {sum(labels)}")
print(f"  No-match (False): {len(labels) - sum(labels)}")
print(f"  Balance: {sum(labels)/len(labels):.2%} positive")

In [ ]:
# TinyCLIP Scorer Implementation
print("Loading CLIP model...")

# Try to load TinyCLIP, fallback to OpenCLIP ViT-B/32 if failed
try:
    # Attempt to load TinyCLIP from HuggingFace hub
    model, preprocess, tokenizer = open_clip.create_model_and_transforms(
        "hf-hub:microsoft/TinyCLIP-ViT-8M-16-Text-3M-YFCC15M"
    )
    print("Successfully loaded TinyCLIP model")
except Exception as e:
    print(f"Failed to load TinyCLIP: {e}")
    print("Falling back to OpenCLIP ViT-B/32...")

    model, preprocess, tokenizer = open_clip.create_model_and_transforms(
        "ViT-B-32", 
        pretrained="laion2b_s34b_b79k"
    )
    print("Successfully loaded OpenCLIP ViT-B/32")

# Move model to device
model = model.to(device)
model.eval()

print(f"Model loaded on: {device}")

In [ ]:
def clip_embed(image: Image.Image, caption: str) -> float:
    """
    Compute cosine similarity between image and text embeddings.
    
    Args:
        image: PIL Image
        caption: Text string
    
    Returns:
        Cosine similarity score between normalized embeddings
    """
    with torch.no_grad():
        # Preprocess image
        image_tensor = preprocess(image).unsqueeze(0).to(device)
        
        # Tokenize text properly using open_clip tokenizer
        text_tokens = open_clip.tokenize([caption]).to(device)
        
        # Get embeddings
        image_features = model.encode_image(image_tensor)
        text_features = model.encode_text(text_tokens)
        
        # Normalize embeddings
        image_features = F.normalize(image_features, dim=-1)
        text_features = F.normalize(text_features, dim=-1)
        
        # Compute cosine similarity
        similarity = (image_features @ text_features.T).item()
        
    return similarity

# Test the embedding function
print("\nTesting CLIP embedding function...")
test_pair = val_pairs[0]
test_image = load_image_from_pair(test_pair)
test_similarity = clip_embed(test_image, test_pair['caption'])
print(f"Sample similarity score: {test_similarity:.4f} (Expected match: {test_pair['is_match']})")

# Test on a few more samples
print("\nTesting on more samples:")
for i in range(3):
    pair = val_pairs[i]
    image = load_image_from_pair(pair)
    similarity = clip_embed(image, pair['caption'])
    print(f"Sample {i+1}: similarity={similarity:.4f}, match={pair['is_match']}")

In [ ]:
# Calibration: Fit logistic regression to get alpha, beta parameters
print("Calibrating scorer with logistic regression...")

# Use first 200 samples for calibration
tune_slice = val_pairs[:200]
print(f"Using {len(tune_slice)} samples for calibration")

# Compute similarities for calibration
similarities = []
ground_truths = []

print("Computing similarities for calibration...")
for pair in tqdm(tune_slice, desc="Calibration"):
    image = load_image_from_pair(pair)
    similarity = clip_embed(image, pair['caption'])
    similarities.append(similarity)
    ground_truths.append(int(pair['is_match']))

similarities = np.array(similarities).reshape(-1, 1)
ground_truths = np.array(ground_truths)

# Fit logistic regression: sigmoid(alpha * cosine + beta)
lr = LogisticRegression()
lr.fit(similarities, ground_truths)

# Extract alpha and beta
alpha = lr.coef_[0][0]  # Coefficient for similarity
beta = lr.intercept_[0]  # Intercept

print(f"Calibration complete!")
print(f"   Alpha (slope): {alpha:.4f}")
print(f"   Beta (intercept): {beta:.4f}")

# Test calibration
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

test_similarities = similarities[:5].flatten()
test_labels = ground_truths[:5]
calibrated_probs = sigmoid(alpha * test_similarities + beta)

print(f"\nCalibration test:")
for i in range(5):
    print(f"  Sim: {test_similarities[i]:.4f} → Prob: {calibrated_probs[i]:.4f}, True: {test_labels[i]}")

In [ ]:
from scipy.ndimage import label

# BlackBox API Implementation
import editdistance  # For Levenshtein distance

class BlackBoxAPI:
    """
    Black-box API for the VLM scorer with query budget tracking and cost calculation.
    """
    
    def __init__(self, alpha: float, beta: float, q_max: int = 200):
        """
        Initialize the black-box API.
        
        Args:
            alpha: Logistic regression slope parameter
            beta: Logistic regression intercept parameter  
            q_max: Maximum queries allowed per sample
        """
        self.alpha = alpha
        self.beta = beta
        self.q_max = q_max
        self.query_count = 0
        
    def score(self, image_uint8: np.ndarray, caption_str: str) -> float:
        """
        Score image-caption pair and return probability.
        
        Args:
            image_uint8: Image as uint8 numpy array (H, W, C)
            caption_str: Caption string
            
        Returns:
            Probability in [0, 1] using sigmoid(alpha * cosine + beta)
        """
        if self.query_count >= self.q_max:
            raise RuntimeError(f"Query budget exceeded! Used {self.query_count}/{self.q_max}")
            
        # Convert numpy array to PIL Image
        image_pil = Image.fromarray(image_uint8)
        
        # Get cosine similarity
        cosine_sim = clip_embed(image_pil, caption_str)
        
        # Apply calibrated sigmoid
        logit = self.alpha * cosine_sim + self.beta
        probability = 1 / (1 + np.exp(-logit))
        
        self.query_count += 1
        
        return probability
    
    def reset_query_count(self):
        """Reset query counter for new sample."""
        self.query_count = 0
        
    def get_remaining_queries(self) -> int:
        """Get remaining query budget."""
        return self.q_max - self.query_count

# Cost Functions
def token_edit_cost(original: str, modified: str) -> int:
    """
    Compute token-level Levenshtein distance using CLIP tokenizer.
    
    Args:
        original: Original caption
        modified: Modified caption
        
    Returns:
        Number of token edits (insertions, deletions, substitutions)
    """
    # Use CLIP tokenizer for more accurate tokenization
    orig_tokens = open_clip.tokenize([original], context_length=77)[0].numpy()
    mod_tokens = open_clip.tokenize([modified], context_length=77)[0].numpy()
    
    # Remove padding tokens (0s) and special tokens for fair comparison
    # Keep only actual content tokens
    orig_tokens = orig_tokens[orig_tokens != 0]
    mod_tokens = mod_tokens[mod_tokens != 0]
    
    return editdistance.eval(orig_tokens.tolist(), mod_tokens.tolist())

def pixel_edit_cost(original: np.ndarray, modified: np.ndarray) -> int:
    """
    Compute number of changed pixels with reduced cost for continuous regions.
    
    Args:
        original: Original image as uint8 numpy array
        modified: Modified image as uint8 numpy array
        
    Returns:
        Adjusted cost based on number of changed pixels, with reduced cost for continuous regions.
    """
    # Find the difference mask
    diff_mask = np.any(original != modified, axis=-1)
    
    # Label connected components in the difference mask
    labeled_regions, num_features = label(diff_mask)
    
    # Count pixels in each connected region
    total_cost = 0
    for region_id in range(1, num_features + 1):
        region_size = np.sum(labeled_regions == region_id)
        if region_size > 0:
            # Full cost for the first pixel, half cost for the rest
            total_cost += 1 + (region_size - 1) * 0.5
    
    return int(total_cost)

# Test the BlackBox API
print("Testing BlackBox API...")

# Initialize API with calibrated parameters
api = BlackBoxAPI(alpha, beta, q_max=200)

# Test on a sample
test_pair = val_pairs[0]
test_image = load_image_from_pair(test_pair)
test_image_uint8 = np.array(test_image)

# Get score
score = api.score(test_image_uint8, test_pair['caption'])
print(f"API Score: {score:.4f} (Expected match: {test_pair['is_match']})")
print(f"Queries used: {api.query_count}/{api.q_max}")

# Test cost functions
original_caption = "A cat sitting on a mat"
modified_caption = "A dog standing on a rug" 
token_cost = token_edit_cost(original_caption, modified_caption)
print(f"\nToken edit cost example:")
print(f"  Original: '{original_caption}'")  
print(f"  Modified: '{modified_caption}'")
print(f"  Cost: {token_cost} token edits")

# Test pixel cost (create a simple modification)
original_img = np.zeros((100, 100, 3), dtype=np.uint8)
modified_img = original_img.copy()
modified_img[10:20, 10:20] = 255  # Change a 10x10 region
pixel_cost = pixel_edit_cost(original_img, modified_img)
print(f"\nPixel edit cost example:")
print(f"  Modified {pixel_cost} pixels in 100x100 image")

## Your Task: Implement Adversarial Attacks

### Attack Function Template

Replace the trivial baseline in the `attack()` function with sophisticated adversarial attacks:

```python
def attack(image_np_uint8, caption_str, api, budgets):
    # Your attack implementation here!
    # You can modify:
    # - Caption tokens (text modifications)  
    # - Image pixels (visual modifications)
    # - Both (multimodal attack)
    
    # Stay within budgets:
    # - budgets['T_MAX'] = 10  token edits
    # - budgets['P_MAX'] = 100 pixel edits
    # - budgets['Q_MAX'] = 100 queries
    
    return {
        'success': success,      # bool: did you flip the decision?
        'image': final_image,    # np.array: attacked image
        'caption': final_caption, # str: attacked caption  
        'token_cost': token_cost, # int: tokens changed
        'pixel_cost': pixel_cost, # int: pixels changed
        'query_cost': query_cost  # int: API calls made
    }
```

### Attack Strategies to Consider

- **Text Attacks**: Synonym replacement, word insertion/deletion, semantic paraphrasing
- **Image Attacks**: Adversarial noise, targeted pixel modifications, patch attacks
- **Query Optimization**: Gradient-free optimization, genetic algorithms, hill climbing
- **Multimodal**: Combined text+image attacks for maximum effectiveness

### Evaluation Metrics

Your attack will be scored as: **ASR - 0.5×ANC - 0.1×(AQ/Q_MAX)**

- **ASR**: Attack Success Rate (higher is better)
- **ANC**: Average Number of Changes (lower is better) 
- **AQ**: Average Queries (lower is better)

### Next Steps

1. **Implement your attack** in the `attack()` function above
2. **Test locally** using the evaluation framework  
3. **Run on full dataset** by changing `max_samples=None`

Good luck! 

### <span style="color:red">**FILL THIS CODE BLOCK**</span>

In [ ]:
def attack(image_np_uint8: np.ndarray, caption_str: str, api: BlackBoxAPI, budgets: dict) -> dict:
    """
    Multimodal attack with minimal text substitution (non-match→match) and
    single-patch image search (match→non-match), honoring token/pixel/query budgets.
    Changes ensure at most one image patch is applied to avoid pixel budget overruns.
    """
    # Originals
    original_image = image_np_uint8.copy()
    original_caption = caption_str

    # Initial score and decision
    try:
        original_score = api.score(original_image, original_caption)
    except RuntimeError as e:
        return {
            'success': False,
            'image': original_image,
            'caption': original_caption,
            'token_cost': 0,
            'pixel_cost': 0,
            'query_cost': api.query_count,
            'error': str(e),
            'original_score': None,
            'final_score': None,
        }

    original_decision = (original_score > 0.5)
    target_increase = not original_decision  # If currently match, we want to decrease; else increase

    H, W, C = original_image.shape

    # Helper: connected-region pixel cost for square patch of size s
    def cost_for_size(s: int) -> int:
        return int(1 + 0.5 * (s * s - 1))

    # Prepare outputs
    working_image = original_image
    working_caption = original_caption
    best_score = original_score

    # 1) Text substitution: only try when we need to increase probability
    if target_increase and budgets.get('T_MAX', 0) > 0 and api.get_remaining_queries() > 1:
        text_candidates = [
            f"a photo of {original_caption}",
            f"a picture of {original_caption}",
            f"photo of {original_caption}",
            f"{original_caption} in a photo",
            f"{original_caption}, high quality",
        ]

        for cand in text_candidates:
            if api.get_remaining_queries() <= 1:
                break
            # Enforce token budget per lab rules
            if token_edit_cost(original_caption, cand) > budgets['T_MAX']:
                continue
            try:
                s = api.score(working_image, cand)
            except RuntimeError:
                break
            # Keep best caption that increases score
            if s > best_score:
                best_score = s
                working_caption = cand
            # Early stop on decision flip
            if (s > 0.5) != original_decision:
                best_score = s
                working_caption = cand
                # Finalize early; avoid image queries
                final_image = working_image
                final_caption = working_caption
                final_score = best_score
                token_cost = token_edit_cost(original_caption, final_caption)
                pixel_cost = pixel_edit_cost(original_image, final_image)
                query_cost = api.query_count
                success = True
                return {
                    'success': success,
                    'image': final_image,
                    'caption': final_caption,
                    'token_cost': token_cost,
                    'pixel_cost': pixel_cost,
                    'query_cost': query_cost,
                    'original_score': original_score,
                    'final_score': final_score,
                }

    # 2) Gradient-guided targeted patch proposal (select pixels that maximize desired change)
    #    Compute gradient of calibrated logit wrt preprocessed input; pick a connected square
    #    around the max-saliency pixel; map region back to original image; test 0/255 fills.
    try:
        # Build 224x224 tensor with grad
        target_size = 224
        resized = Image.fromarray(working_image).resize((target_size, target_size), Image.BICUBIC)
        img_224 = torch.from_numpy(np.array(resized)).permute(2, 0, 1).float() / 255.0
        img_224 = img_224.unsqueeze(0).to(device).requires_grad_(True)

        # Get normalization from preprocess if available, else fallback to CLIP defaults
        norm = None
        if hasattr(preprocess, 'transforms'):
            for t in getattr(preprocess, 'transforms', []):
                if hasattr(t, 'mean') and hasattr(t, 'std'):
                    norm = t
                    break
        if norm is not None:
            mean = torch.tensor(norm.mean, device=device).view(1, 3, 1, 1)
            std = torch.tensor(norm.std, device=device).view(1, 3, 1, 1)
        else:
            mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1, 3, 1, 1)
            std = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1, 3, 1, 1)

        img_norm = (img_224 - mean) / std

        # Encode with gradient
        text_tokens = open_clip.tokenize([working_caption]).to(device)
        image_features = model.encode_image(img_norm)
        text_features = model.encode_text(text_tokens)
        image_features = F.normalize(image_features, dim=-1)
        text_features = F.normalize(text_features, dim=-1)
        cosine = (image_features @ text_features.T)[0, 0]
        logit = alpha * cosine + beta

        objective = logit if target_increase else -logit
        objective.backward()
        saliency = img_224.grad.abs().sum(dim=1)[0]  # (224,224)

        # Select top saliency location
        sal_np = saliency.detach().cpu().numpy()
        y_max, x_max = np.unravel_index(np.argmax(sal_np), sal_np.shape)

        # Choose square size to fit pixel budget after mapping back
        def apply_region(image_np: np.ndarray, x0: int, y0: int, w: int, h: int, color: tuple) -> np.ndarray:
            out = image_np.copy()
            x1 = min(image_np.shape[1], x0 + w)
            y1 = min(image_np.shape[0], y0 + h)
            out[y0:y1, x0:x1, 0] = color[0]
            out[y0:y1, x0:x1, 1] = color[1]
            out[y0:y1, x0:x1, 2] = color[2]
            return out

        # Map region from 224 to original
        ratio_x = W / target_size
        ratio_y = H / target_size

        # Start with s_224=10 and scale; shrink until pixel cost <= P_MAX
        s_224 = 10
        def region_cost_for_s(s224: int) -> int:
            s_w = max(1, int(round(s224 * ratio_x)))
            s_h = max(1, int(round(s224 * ratio_y)))
            x0_224 = max(0, min(target_size - s224, x_max - s224 // 2))
            y0_224 = max(0, min(target_size - s224, y_max - s224 // 2))
            x0 = int(round(x0_224 * ratio_x))
            y0 = int(round(y0_224 * ratio_y))
            cand = apply_region(working_image, x0, y0, s_w, s_h, (255, 255, 255))
            return pixel_edit_cost(working_image, cand)

        while s_224 > 1 and region_cost_for_s(s_224) > budgets.get('P_MAX', 0):
            s_224 -= 1

        if s_224 > 0 and region_cost_for_s(s_224) <= budgets.get('P_MAX', 0) and remaining_queries() > 1:
            s_w = max(1, int(round(s_224 * ratio_x)))
            s_h = max(1, int(round(s_224 * ratio_y)))
            x0_224 = max(0, min(target_size - s_224, x_max - s_224 // 2))
            y0_224 = max(0, min(target_size - s_224, y_max - s_224 // 2))
            x0 = int(round(x0_224 * ratio_x))
            y0 = int(round(y0_224 * ratio_y))

            # Try two extreme fills and keep the better in desired direction
            for color in [(255, 255, 255), (0, 0, 0)]:
                if remaining_queries() <= 1:
                    break
                candidate = apply_region(working_image, x0, y0, s_w, s_h, color)
                s = api.score(candidate, working_caption)
                improved = (s > best_score) if target_increase else (s < best_score)
                if improved:
                    best_score = s
                    working_image = candidate
                if (s > 0.5) != original_decision:
                    final_image = candidate
                    final_caption = working_caption
                    final_score = s
                    token_cost = token_edit_cost(original_caption, final_caption)
                    pixel_cost = pixel_edit_cost(original_image, final_image)
                    query_cost = api.query_count
                    success = True
                    return {
                        'success': success,
                        'image': final_image,
                        'caption': final_caption,
                        'token_cost': token_cost,
                        'pixel_cost': pixel_cost,
                        'query_cost': query_cost,
                        'original_score': original_score,
                        'final_score': final_score,
                    }
    except Exception:
        # If gradient step fails, fall back to simple search below
        pass

    # 2) Image patch search: single patch only, based on the same base image
    # Pick a patch size that respects remaining pixel budget
    patch_size = 10
    while patch_size > 1 and cost_for_size(patch_size) > budgets.get('P_MAX', 0):
        patch_size -= 1

    def apply_patch(img: np.ndarray, x: int, y: int, s: int, color: tuple) -> np.ndarray:
        patched = img.copy()
        patched[y:y+s, x:x+s, 0] = color[0]
        patched[y:y+s, x:x+s, 1] = color[1]
        patched[y:y+s, x:x+s, 2] = color[2]
        return patched

    def remaining_queries() -> int:
        return min(15, budgets['Q_MAX']) - api.query_count

    if patch_size > 0 and cost_for_size(patch_size) <= budgets.get('P_MAX', 0) and remaining_queries() > 1:
        # Very small search to cap queries
        positions = [
            ((W - patch_size) // 2, (H - patch_size) // 2),  # center
            (0, 0),                                          # top-left
        ]

        # Two strong seeds only
        initial_colors = [
            (255, 255, 255),  # white
            (0, 0, 0),        # black
        ]
        channel_steps = [32]  # one step

        base_img = working_image  # single-patch candidates only
        best_patch_image = None
        best_patch_score = best_score
        flipped = False

        def clamp(v):
            return int(max(0, min(255, v)))

        for (x, y) in positions:
            if remaining_queries() <= 1 or flipped:
                break

            # Try initial seeds
            seeds = list(initial_colors)
            for seed in seeds:
                if remaining_queries() <= 1 or flipped:
                    break
                candidate = apply_patch(base_img, x, y, patch_size, seed)
                try:
                    s = api.score(candidate, working_caption)
                except RuntimeError:
                    break

                improved = (s > best_patch_score) if target_increase else (s < best_patch_score)
                if improved:
                    best_patch_score = s
                    best_patch_image = candidate
                if (s > 0.5) != original_decision:
                    best_patch_score = s
                    best_patch_image = candidate
                    flipped = True
                    break

                # Hill-climb around this seed by adjusting channels
                r, g, b = seed
                for step in channel_steps:
                    if remaining_queries() <= 1 or flipped:
                        break
                    for dr, dg, db in [(step,0,0),(-step,0,0),(0,step,0),(0,-step,0),(0,0,step),(0,0,-step)]:
                        if remaining_queries() <= 1 or flipped:
                            break
                        trial = (clamp(r+dr), clamp(g+dg), clamp(b+db))
                        candidate2 = apply_patch(base_img, x, y, patch_size, trial)
                        try:
                            s2 = api.score(candidate2, working_caption)
                        except RuntimeError:
                            break
                        improved2 = (s2 > best_patch_score) if target_increase else (s2 < best_patch_score)
                        if improved2:
                            best_patch_score = s2
                            best_patch_image = candidate2
                        if (s2 > 0.5) != original_decision:
                            best_patch_score = s2
                            best_patch_image = candidate2
                            flipped = True
                            break

        if best_patch_image is not None:
            working_image = best_patch_image
            best_score = best_patch_score
            if flipped:
                # Early finalize after image success
                final_image = working_image
                final_caption = working_caption
                final_score = best_score
                token_cost = token_edit_cost(original_caption, final_caption)
                pixel_cost = pixel_edit_cost(original_image, final_image)
                query_cost = api.query_count
                success = True
                return {
                    'success': success,
                    'image': final_image,
                    'caption': final_caption,
                    'token_cost': token_cost,
                    'pixel_cost': pixel_cost,
                    'query_cost': query_cost,
                    'original_score': original_score,
                    'final_score': final_score,
                }

    final_image = working_image
    final_caption = working_caption

    # Final score
    try:
        final_score = api.score(final_image, final_caption)
    except RuntimeError:
        final_score = best_score

    success = (final_score > 0.5) != original_decision

    # Costs
    token_cost = token_edit_cost(original_caption, final_caption)
    pixel_cost = pixel_edit_cost(original_image, final_image)
    query_cost = api.query_count

    # Enforce budget validity (evaluation will also check)
    if token_cost > budgets.get('T_MAX', 0) or pixel_cost > budgets.get('P_MAX', 0) or query_cost > budgets.get('Q_MAX', 0):
        success = False

    return {
        'success': success,
        'image': final_image,
        'caption': final_caption,
        'token_cost': token_cost,
        'pixel_cost': pixel_cost,
        'query_cost': query_cost,
        'original_score': original_score,
        'final_score': final_score,
    }

print("Attack function defined (Text+Image baseline, single-patch and early-returns)")
print("   Ensures at most one image patch; fewer queries; early exit on success.")

# Test the attack function
print("\nTesting attack function...")
test_pair = val_pairs[0]
test_image = np.array(load_image_from_pair(test_pair))

# Create fresh API instance  
test_api = BlackBoxAPI(alpha, beta, q_max=100)

In [ ]:
# Evaluation Framework
def evaluate_attack(val_pairs: list, attack_function, alpha: float, beta: float, budgets: dict, max_samples: int = None):
    """
    Evaluate attack function on validation pairs.
    
    Args:
        val_pairs: List of validation pairs
        attack_function: Attack function to evaluate
        alpha, beta: Calibrated parameters
        budgets: Attack budgets dictionary
        max_samples: Limit number of samples (None = all)
        
    Returns:
        Dictionary with evaluation metrics
    """
    
    print(f"Starting evaluation...")
    
    # Limit samples if specified
    eval_pairs = val_pairs[:max_samples] if max_samples else val_pairs
    print(f"Evaluating on {len(eval_pairs)} samples")
    
    results = []
    total_success = 0
    total_token_cost = 0
    total_pixel_cost = 0
    total_query_cost = 0
    
    for i, pair in enumerate(tqdm(eval_pairs, desc="Attacking")):
        # Create fresh API instance for each sample
        api = BlackBoxAPI(alpha, beta, q_max=budgets['Q_MAX'])
        
        # Load image
        image = np.array(load_image_from_pair(pair))
        caption = pair['caption']
        
        try:
            # Run attack
            result = attack_function(image, caption, api, budgets)
            
            # Validate budget constraints
            budget_valid = (
                result['token_cost'] <= budgets['T_MAX'] and
                result['pixel_cost'] <= budgets['P_MAX'] and  
                result['query_cost'] <= budgets['Q_MAX']
            )
            
            if not budget_valid:
                print(f"Sample {i}: Budget violation!")
                print(f"   Tokens: {result['token_cost']}/{budgets['T_MAX']}")
                print(f"   Pixels: {result['pixel_cost']}/{budgets['P_MAX']}")  
                print(f"   Queries: {result['query_cost']}/{budgets['Q_MAX']}")
                result['success'] = False  # Invalid attacks count as failures
            
            results.append(result)
            
            if result['success']:
                total_success += 1
            total_token_cost += result['token_cost']
            total_pixel_cost += result['pixel_cost']
            total_query_cost += result['query_cost']
            
        except Exception as e:
            print(f"Sample {i}: Attack failed with error: {e}")
            # Add failed result
            results.append({
                'success': False,
                'token_cost': budgets['T_MAX'],  # Penalize failures
                'pixel_cost': budgets['P_MAX'], 
                'query_cost': budgets['Q_MAX'],
                'error': str(e)
            })
    
    # Calculate metrics
    n_samples = len(results)
    asr = total_success / n_samples  # Attack Success Rate
    anc = (10*total_token_cost + total_pixel_cost) / n_samples  # Average Number of Changes  
    aq = total_query_cost / n_samples  # Average Queries
    
    # Final score: ASR - 0.5*ANC - 0.1*(AQ/Q_MAX)
    score = asr - 0.5 * (anc / (10*budgets['T_MAX'] + budgets['P_MAX'])) - 0.1 * (aq / budgets['Q_MAX'])
    
    evaluation_result = {
        'ASR': asr,
        'ANC': anc, 
        'AQ': aq,
        'Score': score,
        'n_samples': n_samples,
        'total_success': total_success,
        'avg_token_cost': total_token_cost / n_samples,
        'avg_pixel_cost': total_pixel_cost / n_samples,
        'budgets': budgets,
        'results': results
    }
    
    return evaluation_result

# Run Evaluation
print("Running evaluation on validation set...")

# Define attack budgets
attack_budgets = {
    'T_MAX': 10,     # Maximum token edits per sample
    'P_MAX': 100,   # Maximum pixel edits per sample  
    'Q_MAX': 100    # Maximum queries per sample
}

# Evaluate on subset first (faster for testing)
print("Running on first 50 samples for quick testing...")
eval_result = evaluate_attack(
    val_pairs=val_pairs, 
    attack_function=attack,
    alpha=alpha,
    beta=beta, 
    budgets=attack_budgets,
    max_samples=50  # Quick test on 50 samples
)

# Print results
print(f"\nEVALUATION RESULTS (50 samples):")
print(f"{'='*50}")
print(f"Attack Success Rate (ASR): {eval_result['ASR']:.1%}")
print(f"Average Number of Changes (ANC): {eval_result['ANC']:.2f}")  
print(f"Average Queries (AQ): {eval_result['AQ']:.1f}")
print(f"Final Score: {eval_result['Score']:.4f}")
print(f"{'='*50}")
print(f"Budget Usage:")
print(f"  Avg Token Cost: {eval_result['avg_token_cost']:.2f}/{attack_budgets['T_MAX']}")
print(f"  Avg Pixel Cost: {eval_result['avg_pixel_cost']:.2f}/{attack_budgets['P_MAX']}")  
print(f"  Avg Query Cost: {eval_result['AQ']:.1f}/{attack_budgets['Q_MAX']}")
print(f"\nNOTE: This is a trivial baseline (0% ASR expected)")
print(f"Students should implement sophisticated attacks to improve ASR!")

In [ ]:
# Full Evaluation (Uncomment when ready to test your attack)

def run_full_evaluation():
    """Run evaluation on all 1000 validation samples."""
    print("Running FULL evaluation on all 1000 validation samples...")
    print("This may take several minutes depending on your attack implementation.")
    
    full_result = evaluate_attack(
        val_pairs=val_pairs,
        attack_function=attack, 
        alpha=alpha,
        beta=beta,
        budgets=attack_budgets,
        max_samples=None  # All samples
    )
    
    print(f"\nFINAL EVALUATION RESULTS:")
    print(f"{'='*60}")
    print(f"Attack Success Rate (ASR): {full_result['ASR']:.1%}")
    print(f"Average Number of Changes (ANC): {full_result['ANC']:.2f}")
    print(f"Average Queries (AQ): {full_result['AQ']:.1f}")
    print(f"Final Score: {full_result['Score']:.4f}")
    print(f"{'='*60}")
    
    return full_result

# Uncomment the line below when ready to run full evaluation:
full_results = run_full_evaluation()

print("To run full evaluation on all 1000 samples:")
print("Uncomment: full_results = run_full_evaluation()")
print("\nCurrent status: Framework ready for student implementations!")